# 임베딩과 의미 유사도

단어가 다르더라도 뜻이 비슷한 문장을 숫자로 비교할 수 있을까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 두 숫자 벡터로 표기 읽기

벡터는 순서가 있는 숫자 목록입니다. `[3, 4]`와 `[4, 3]`을 두 행으로 담습니다. `shape`는 행과 열의 수를 반환합니다. `axis=1`은 열 방향으로 각 행의 값을 묶어 벡터마다 길이 하나를 계산하라는 뜻입니다. `keepdims=True`는 결과를 `(2,)`가 아닌 `(2, 1)` 열 모양으로 유지하여 각 행을 자기 길이로 나눌 수 있게 합니다. `ravel()`은 출력할 때 이 열 모양을 1차원 `[5, 5]`로 펼칩니다. 각 행을 길이로 나눈 뒤 `T`로 행과 열을 바꾸고 `@`를 적용하면 모든 행 쌍의 점곱 표가 반환됩니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import numpy as np

toy = np.array([[3.0, 4.0], [4.0, 3.0]])
lengths = np.linalg.norm(toy, axis=1, keepdims=True)
normalized_toy = toy / lengths
toy_scores = normalized_toy @ normalized_toy.T
print("shape:", toy.shape)
print("lengths:", lengths.ravel())
print("normalized:\n", normalized_toy)
print("cosine table:\n", toy_scores)

shape: (2, 2)
lengths: [5. 5.]
normalized:
 [[0.6 0.8]
 [0.8 0.6]]
cosine table:
 [[1.   0.96]
 [0.96 1.  ]]


## 다국어 문장 임베딩

벡터는 순서가 있는 숫자 목록입니다. 문장 네 개가 **입력**입니다. `model.encode()`는 각 문장을 384개의 실수로 바꾸어 NumPy 배열로 **반환**합니다. `shape`의 `(4, 384)`는 4개 행이 문장 네 개에, 384개 열이 각 문장의 숫자 표현에 대응한다는 뜻입니다. `vectors[0, :5]`는 구조를 확인하려고 첫 문장의 앞 다섯 값만 고릅니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID, revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42")
sentences = [
    "주문한 상품을 취소하고 싶습니다.",
    "구매를 취소하려면 어떻게 하나요?",
    "내일 서울은 비가 옵니다.",
    "I want to cancel my order.",
]
vectors = model.encode(sentences, normalize_embeddings=True)
print("shape:", vectors.shape)
print("첫 문장 앞 5개 값:", np.round(vectors[0, :5], 4))

shape: (4, 384)
첫 문장 앞 5개 값: [ 0.0091  0.0092  0.0109 -0.0262 -0.0082]


## 코사인 유사도 비교

실제 문장 벡터는 길이 1로 정규화되어 있습니다. `vectors.T`는 행과 열을 바꾸고, `@`는 행렬 곱을 수행합니다. 따라서 `vectors @ vectors.T`는 모든 문장 쌍의 점곱, 곧 코사인 유사도를 4×4 배열로 반환합니다. 0번 문장을 자기 자신과 비교한 값도 1이므로 후보 순서를 만들 때는 0번을 명시적으로 제외합니다. `np.argsort(-scores[0])`는 0번 행의 유사도를 큰 값부터 정렬한 인덱스를 반환하고, 뒤의 `[1:]`이 첫 인덱스 0을 제외합니다. `nearest`는 NumPy 배열이며 `nearest.tolist()`는 출력하기 쉬운 Python 목록으로 변환합니다. 출력의 `0 vs i`는 항상 0번 취소 문장을 기준으로 읽습니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
scores = vectors @ vectors.T
for i in range(1, len(sentences)):
    print(f"0 vs {i}: {scores[0, i]:.4f} | {sentences[i]}")
nearest = np.argsort(-scores[0])[1:]
print("\n유사도 순서:", nearest.tolist())

0 vs 1: 0.8024 | 구매를 취소하려면 어떻게 하나요?
0 vs 2: 0.1044 | 내일 서울은 비가 옵니다.
0 vs 3: 0.8987 | I want to cancel my order.

유사도 순서: [3, 1, 2]


## 단어 하나 바꾸기

'취소'를 '반품'으로 바꿔 유사도가 어떻게 달라지는지 봅니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
changed = model.encode(["주문한 상품을 반품하고 싶습니다."], normalize_embeddings=True)[0]
print("취소 문장 vs 반품 문장:", round(float(vectors[0] @ changed), 4))
print("취소 문장 vs 날씨 문장:", round(float(vectors[0] @ vectors[2]), 4))

취소 문장 vs 반품 문장: 0.5931
취소 문장 vs 날씨 문장: 0.1044


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [5]:
assert toy.shape == (2, 2) and np.allclose(lengths.ravel(), [5.0, 5.0])
assert np.allclose(np.diag(toy_scores), [1.0, 1.0])
assert vectors.shape == (4, 384) and np.isfinite(vectors).all()
assert scores[0, 3] > scores[0, 1] > scores[0, 2]
assert float(vectors[0] @ changed) > scores[0, 2]
print("검증: 2차원 표기, 384차원 벡터 구조와 본문의 유사도 순서를 확인했습니다.")

검증: 2차원 표기, 384차원 벡터 구조와 본문의 유사도 순서를 확인했습니다.
